In [0]:
import json
from functools import reduce
from pyspark.sql.functions import (
    col, regexp_replace, trim, lit, when, length, 
    row_number, desc, lower, current_timestamp
)
from pyspark.sql.window import Window

# 1. Cargar Credenciales
try:
    with open("aws_secrets.json", "r") as f:
        config = json.load(f)
    print("✅ Credenciales cargadas.")
except:
    print("❌ Error cargando secretos.")

aws_access_key = config['aws_access_key']
aws_secret_key = config['aws_secret_key']
bucket_name = "bronce-scrap-date" 

# ==========================================
# FUNCIÓN DE NORMALIZACIÓN (Con Credenciales Inyectadas)
# ==========================================
def normalizar_tabla(fuente, key, secret):
    ruta_bronze = f"s3a://{bucket_name}/bronze/{fuente}/"
    print(f"📖 Procesando fuente: {fuente}...")
    
    try:
        # EL FIX: Pasamos las llaves AQUÍ, igual que hicimos en Bronze
        df = spark.read.format("delta") \
            .option("fs.s3a.access.key", key) \
            .option("fs.s3a.secret.key", secret) \
            .option("fs.s3a.endpoint", "s3.amazonaws.com") \
            .load(ruta_bronze)
            
        cols = df.columns
        def safe_col(nombre_col):
            return col(nombre_col) if nombre_col in cols else lit(None).cast("string")

        # Limpieza previa
        precio_sucio = regexp_replace(safe_col("price"), r"[^0-9]", "")
        area_sucia = regexp_replace(safe_col("area"), r"[^0-9]", "")
        
        # --- LÓGICA DE TRANSFORMACIÓN ---
        df_clean = df.select(
            col("id_inmueble").alias("id_original"),
            col("bronze_ingested_at").alias("fecha_extraccion"), 
            lit(fuente).alias("fuente"), 
            col("url").alias("url_detalle"),
            
            # Limpieza de Ubicación (Anti-NaN)
            when(
                safe_col("location").isNull() | 
                (length(trim(safe_col("location"))) < 3) | 
                (lower(safe_col("location")) == "nan"), 
                lit("Sin Barrio")
            ).otherwise(trim(safe_col("location"))).alias("barrio"),
            
            safe_col("title").alias("titulo"),
            
            # Conversión de Precio
            when(length(precio_sucio) > 0, precio_sucio.cast("long")) \
                .otherwise(lit(None)).alias("precio_num"),
            
            # Conversión de Área
            when(length(area_sucia) > 0, area_sucia.cast("int")) \
                .otherwise(lit(None)).alias("area_m2")
        )
        
        # --- FILTROS DE CALIDAD ---
        df_final = df_clean.filter(
            (col("precio_num").isNotNull()) & 
            (col("precio_num") > 20000000) & 
            (col("precio_num") < 20000000000)
        )
        
        return df_final

    except Exception as e:
        print(f"⚠️ Saltando {fuente} (Error o sin datos): {str(e)[0:100]}...")
        return None

# ==========================================
# EJECUCIÓN (Usando las variables key/secret)
# ==========================================
print("🔄 Iniciando Unificación Silver...")

fuentes = ["fincaraiz", "mercadolibre", "metrocuadrado", "bancolombia_tu360", "facebook", "ciencuadras"]
dataframes_limpios = []

for f in fuentes:
    # Pasamos las llaves a la función
    df_temp = normalizar_tabla(f, aws_access_key, aws_secret_key)
    if df_temp is not None:
        dataframes_limpios.append(df_temp)

# Unimos todo
if len(dataframes_limpios) > 0:
    df_silver_unificado = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), dataframes_limpios)
    
    # Deduplicación
    ventana = Window.partitionBy("id_original", "fuente").orderBy(desc("fecha_extraccion"))
    
    df_silver_final = df_silver_unificado \
        .withColumn("rank", row_number().over(ventana)) \
        .filter(col("rank") == 1) \
        .drop("rank") \
        .withColumn("silver_processed_at", current_timestamp())
        
    print(f"📉 Registros únicos finales: {df_silver_final.count()}")

    # 1. Guardar Silver (Con Credenciales)
    ruta_silver = f"s3a://{bucket_name}/silver/master_inmuebles/"
    df_silver_final.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("fs.s3a.access.key", aws_access_key) \
        .option("fs.s3a.secret.key", aws_secret_key) \
        .option("fs.s3a.endpoint", "s3.amazonaws.com") \
        .save(ruta_silver)
        
    print(f"🥈 Capa Silver OK: {ruta_silver}")

    # 2. Guardar Gold (Con Credenciales)
    ruta_gold = f"s3a://{bucket_name}/gold/app_consumable/"
    df_silver_final.coalesce(1).write.format("parquet") \
        .mode("overwrite") \
        .option("fs.s3a.access.key", aws_access_key) \
        .option("fs.s3a.secret.key", aws_secret_key) \
        .option("fs.s3a.endpoint", "s3.amazonaws.com") \
        .save(ruta_gold)
        
    print(f"🥇 Capa Gold OK: {ruta_gold}")
    
    display(df_silver_final)

else:
    print("❌ No se encontraron datos para procesar.")

In [0]:
from pyspark.sql.functions import col, regexp_replace, trim, lit, when, length # <--- Agregué length

def normalizar_tabla(fuente, access_key, secret_key):
    # Fíjate que leemos de la carpeta BRONZE (los datos ya transformados a Delta)
    ruta_bronze = f"s3a://{bucket_name}/bronze/{fuente}/"
    
    print(f"📖 Leyendo Bronze: {fuente}...")
    
    try:
        # A. Leemos la tabla Delta (mucho más rápido que leer los JSON)
        df = spark.read.format("delta") \
            .option("fs.s3a.access.key", access_key) \
            .option("fs.s3a.secret.key", secret_key) \
            .option("fs.s3a.endpoint", "s3.amazonaws.com") \
            .load(ruta_bronze)
            
        cols = df.columns
        def safe_col(nombre_col):
            return col(nombre_col) if nombre_col in cols else lit(None).cast("string")

        # --- PREPARACIÓN DE LIMPIEZA ---
        # 1. Limpiamos el precio quitando todo lo que no sea números
        precio_limpio = regexp_replace(col("price"), r"[^0-9]", "")
        
        # 2. Limpiamos el área igual
        area_limpia = regexp_replace(safe_col("area"), r"[^0-9]", "")

        # C. Transformación Final con Protección de Errores
        df_clean = df.select(
            col("id_inmueble").alias("id_original"),
            col("extracted_at").cast("timestamp").alias("fecha_extraccion"),
            col("source_system").alias("fuente"),
            col("url"),
            safe_col("title").alias("titulo"),
            safe_col("location").alias("ubicacion_raw"),
            
            # --- FIX DEL ERROR ---
            # Si la longitud del precio limpio es 0 (estaba vacío o era texto), pon NULL.
            # Si no, conviértelo a Long.
            when(length(precio_limpio) > 0, precio_limpio.cast("long")) \
                .otherwise(lit(None)).alias("precio_num"),
            
            # Hacemos lo mismo con el área
            when(length(area_limpia) > 0, area_limpia.cast("int")) \
                .otherwise(lit(None)).alias("area_m2")
        )
        
        return df_clean

    except Exception as e:
        print(f"⚠️ Saltando {fuente}: {str(e)[0:100]}...")
        return None

In [0]:
from pyspark.sql.functions import col, regexp_replace, length, when, lit, split, trim

def normalizar_tabla(fuente, access_key, secret_key):
    ruta_bronze = f"s3a://{bucket_name}/bronze/{fuente}/"
    print(f"📖 Procesando fuente: {fuente}...")
    
    try:
        df = spark.read.format("delta") \
            .option("fs.s3a.access.key", access_key) \
            .option("fs.s3a.secret.key", secret_key) \
            .option("fs.s3a.endpoint", "s3.amazonaws.com") \
            .load(ruta_bronze)
            
        cols = df.columns
        def safe_col(nombre_col):
            return col(nombre_col) if nombre_col in cols else lit(None).cast("string")

        # Limpieza inicial de precio y area
        precio_sucio = regexp_replace(safe_col("price"), r"[^0-9]", "")
        area_sucia = regexp_replace(safe_col("area"), r"[^0-9]", "")
        
        df_clean = df.select(
            col("id_inmueble").alias("id_original"),
            
            # --- LA LÍNEA QUE FALTABA ---
            # Necesaria para saber cuál dato es el más reciente
            col("bronze_ingested_at").alias("fecha_extraccion"), 
            
            col("source_system").alias("fuente"),
            col("url"),
            
            # Limpieza de Ubicación
            when(safe_col("location").isNull() | (length(safe_col("location")) < 3), "Ubicación Pendiente")
                .otherwise(trim(safe_col("location"))).alias("ubicacion_raw"),
            
            safe_col("title").alias("titulo"),
            
            # Conversión de Precio
            when(length(precio_sucio) > 0, precio_sucio.cast("long")) \
                .otherwise(lit(None)).alias("precio_num"),
            
            # Conversión de Área
            when(length(area_sucia) > 0, area_sucia.cast("int")) \
                .otherwise(lit(None)).alias("area_m2")
        )
        
        # --- FILTROS DE CALIDAD ---
        # 1. Sin nulos críticos
        df_clean = df_clean.filter(col("precio_num").isNotNull() & col("ubicacion_raw").isNotNull())
        
        # 2. Sin Arriendos (Precios < 20 Millones)
        df_clean = df_clean.filter(col("precio_num") > 20000000)
        
        return df_clean

    except Exception as e:
        print(f"⚠️ Saltando {fuente}: {e}")
        return None

In [0]:
print("🔄 Iniciando Unificación de Fuentes...")

dataframes_limpios = []

# 1. Convertimos cada fuente en un DataFrame estandarizado
for fuente in fuentes:
    df_temp = normalizar_tabla(fuente, aws_access_key, aws_secret_key)
    if df_temp is not None:
        dataframes_limpios.append(df_temp)

# 2. Unimos todo (Union All)
if len(dataframes_limpios) > 0:
    # reduce aplica la unión uno por uno a toda la lista
    df_silver_unificado = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), dataframes_limpios)
    
    print(f"📊 Total de registros brutos unificados: {df_silver_unificado.count()}")
else:
    print("❌ No hay datos para procesar.")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

if 'df_silver_unificado' in locals():
    # 1. Definimos la estrategia de deduplicación
    # "Agrupa por ID y Fuente, ordena por fecha (el más nuevo primero)"
    ventana = Window.partitionBy("id_original", "fuente").orderBy(desc("fecha_extraccion"))
    
    # 2. Aplicamos el filtro
    df_silver_final = df_silver_unificado \
        .withColumn("rank", row_number().over(ventana)) \
        .filter(col("rank") == 1) \
        .drop("rank") # Borramos la columna auxiliar
        
    print(f"📉 Registros únicos finales: {df_silver_final.count()}")

    # 3. Guardamos la TABLA MAESTRA en Silver
    ruta_silver = f"s3a://{bucket_name}/silver/master_inmuebles/"
    
    df_silver_final.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("fs.s3a.access.key", aws_access_key) \
        .option("fs.s3a.secret.key", aws_secret_key) \
        .option("fs.s3a.endpoint", "s3.amazonaws.com") \
        .save(ruta_silver)
        
    print("🥈 ¡Capa Silver creada exitosamente!")
    print(f"📂 Ubicación: {ruta_silver}")
    
    # 4. Muestra final
    display(df_silver_final)

In [0]:
# Este código garantiza que en la carpeta 'app_consumable' SIEMPRE haya solo lo último
ruta_produccion = f"s3a://{bucket_name}/gold/app_consumable/"

print(" Limpiando y Publicando versión para la API...")

df_silver_final.coalesce(1).write \
    .format("parquet") \
    .mode("overwrite") \
    .option("fs.s3a.access.key", aws_access_key) \
    .option("fs.s3a.secret.key", aws_secret_key) \
    .option("fs.s3a.endpoint", "s3.amazonaws.com") \
    .save(ruta_produccion)

print("✅ ¡Listo! La API leerá de esta carpeta limpia.")